# Assignment 12: K-Nearest Neighbors

Fill in every `# TODO` and answer every question in the **Your answer** cells. Run all cells before submitting.

## Part A. KNN classification: breast cancer diagnosis

Same data and same 70/30 split as Assignments 8 and 9, so the results are comparable.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import accuracy_score, mean_squared_error

bc = pd.read_csv("https://bryantstats.github.io/math460/data/breast_cancer.csv")
bc = bc.drop(columns=["id", "Unnamed: 32"], errors="ignore")
y = (bc["diagnosis"] == "M").astype(int)
X = bc.drop(columns=["diagnosis"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1, stratify=y)

### Question 1. Scaling matters

a. Train a `KNeighborsClassifier(n_neighbors=5)` on the raw (unscaled) data and report its test accuracy.

b. Train the same model in a pipeline with `StandardScaler()` and report its test accuracy.

c. Look at the ranges of the predictors (printed below). Explain why scaling makes such a difference for KNN.

In [ ]:
# TODO: train `knn_raw` (no scaling) and `knn_scaled` (with StandardScaler), both with k = 5
print(f"k = 5, no scaling:   test accuracy = {accuracy_score(y_test, knn_raw.predict(X_test)):.3f}")
print(f"k = 5, with scaling: test accuracy = {accuracy_score(y_test, knn_scaled.predict(X_test)):.3f}")
print()
print((X.max() - X.min()).sort_values(ascending=False).round(3).head(8))

**Your answer:** *Write your answer here.*

### Question 2. Choosing k

a. Use 5-fold cross-validation **on the training data** to compute the accuracy of the scaled KNN for $k = 1, 2, \dots, 40$, and plot accuracy against $k$.

b. Which $k$ has the highest CV accuracy? Train that model on all the training data and report its test accuracy. Then look at the whole curve: how different are the CV accuracies of the best few values of $k$? Would you really trust the best $k$ over its neighbors?

c. What goes wrong when $k$ is very small (e.g. 1)? When $k$ is very large? Use the words *overfitting* and *underfitting*.

In [ ]:
ks = range(1, 41)
cv_acc = []
for k in ks:
    # TODO: append the mean 5-fold CV accuracy of the scaled KNN with this k

best_k = ks[int(np.argmax(cv_acc))]
plt.plot(ks, cv_acc, marker="o", ms=3)
plt.axvline(best_k, color="gray", ls="--")
plt.xlabel("k"); plt.ylabel("5-fold CV accuracy"); plt.title(f"Best k = {best_k}"); plt.show()

best_knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=best_k)).fit(X_train, y_train)
print(f"k = {best_k}: test accuracy = {accuracy_score(y_test, best_knn.predict(X_test)):.3f}")

**Your answer:** *Write your answer here.*

### Question 3. The curse of dimensionality

KNN treats every predictor as equally important. What happens when we add predictors that carry **no information**?

Add 10, 50, 200 and 1,000 columns of random noise (standard normal) to the data, and compute the test accuracy of the scaled KNN with your best $k$ each time. Explain the result. Would a random forest or LASSO suffer as much? Why or why not?

In [ ]:
rng = np.random.default_rng(0)
for n_noise in [0, 10, 50, 200, 1000]:
    noise_train = rng.normal(size=(len(X_train), n_noise))
    noise_test = rng.normal(size=(len(X_test), n_noise))
    # TODO: stack the noise next to X_train / X_test, train the scaled KNN with best_k, compute the test accuracy
    print(f"{n_noise:>5} noise predictors: test accuracy = {acc:.3f}")

**Your answer:** *Write your answer here.*

## Part B. KNN regression: NBA salaries

KNN can also predict a number: the average $y$ of the $k$ nearest neighbors. Same data and split as Assignment 8.

In [ ]:
nba = pd.read_csv("https://bryantstats.github.io/math460/data/nba_salary2.csv").dropna()
yr = nba["Salary"]
Xr = nba.drop(columns=["Salary"]).select_dtypes("number")
Xr_train, Xr_test, yr_train, yr_test = train_test_split(Xr, yr, test_size=0.3, random_state=1)

### Question 4. KNN regression vs. LASSO

a. Choose $k$ for a scaled `KNeighborsRegressor` by 5-fold cross-validation on the training data (use `scoring="neg_root_mean_squared_error"`; larger is better).

b. Report the test RMSE of the best model and compare it with LASSO's test RMSE from Assignment 8 (about 5.22).

c. Try the option `weights="distance"` (closer neighbors count more). Does it help?

In [ ]:
def cv_rmse(model):
    return -cross_val_score(model, Xr_train, yr_train, cv=5, scoring="neg_root_mean_squared_error").mean()

results = {}
for weights in ["uniform", "distance"]:
    # TODO: for k = 1..40, compute cv_rmse of the scaled KNeighborsRegressor; keep the best k
    print(f"weights = {weights:<9} best k = {k_best:>2}   CV RMSE = {scores[k_best]:.3f}   test RMSE = {test_rmse:.3f}")

**Your answer:** *Write your answer here.*